# 🚀 LakeForge Simple Pipeline Guide

**Welcome!** This notebook is designed for beginners and non-coders.

## What This Pipeline Does:
1. ✅ Creates sample customer data
2. ✅ Validates data quality (checks for errors)
3. ✅ Saves data to Bronze layer (raw storage)
4. ✅ Runs trust checks (ensures data integrity)
5. ✅ Generates a trust report (shows data health)

## How to Use:
**Simply click "Run All" or run each cell one by one from top to bottom.**

No coding knowledge required! Just follow the instructions in each cell.

---

### Prerequisites:
- ✅ LakeForge package installed at `/Workspace/Users/jayarampogakula@gmail.com/lakeforge/`
- ✅ Serverless compute (auto-attached)

### Estimated Time: 2-3 minutes

---

In [0]:
# STEP 1: SETUP
# This cell loads the LakeForge tools we need
# Just click "Run" - no changes needed!

import sys
sys.path.insert(0, '/Workspace/Users/jayarampogakula@gmail.com/lakeforge')

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col

# Initialize Spark
spark = SparkSession.builder.appName("LakeForge-Simple-Pipeline").getOrCreate()

print("✅ Setup complete! Spark is ready.")
print(f"Spark version: {spark.version}")
print("\n➡️ Ready for next step!")

In [0]:
# STEP 2: CREATE SAMPLE DATA
# This creates a small dataset of 10 customers
# No external files needed - data is created in memory

print("Creating sample customer data...\n")

# Sample customer records
sample_data = [
    (1, "john.doe@example.com", "John Doe", "USA", 1500.50),
    (2, "jane.smith@example.com", "Jane Smith", "UK", 2300.75),
    (3, "bob.wilson@example.com", "Bob Wilson", "Canada", 1800.00),
    (4, "alice.brown@example.com", "Alice Brown", "USA", 2100.25),
    (5, "charlie.davis@example.com", "Charlie Davis", "UK", 1950.00),
    (6, "diana.martin@example.com", "Diana Martin", "Germany", 2500.00),
    (7, "frank.lee@example.com", "Frank Lee", "Japan", 1700.50),
    (8, "grace.taylor@example.com", "Grace Taylor", "Australia", 2200.00),
    (9, "henry.clark@example.com", "Henry Clark", "France", 1850.75),
    (10, "iris.walker@example.com", "Iris Walker", "Spain", 2400.00)
]

# Define the structure of our data
schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("email", StringType(), True),
    StructField("name", StringType(), True),
    StructField("country", StringType(), True),
    StructField("total_sales", DoubleType(), True)
])

# Create the DataFrame
df_customers = spark.createDataFrame(sample_data, schema)

print(f"✅ Created {df_customers.count()} customer records\n")
print("Sample data preview:")
display(df_customers)

print("\n➡️ Data created successfully! Ready for next step.")

In [0]:
# STEP 3: SAVE TO BRONZE LAYER
# Bronze = Raw data storage with audit columns
# This adds tracking information (who, when, where)

from lakeforge.bronze import create_bronze_writer
from lakeforge.observability import LakeForgeLogger

logger = LakeForgeLogger(name="simple_pipeline")
logger.info("Starting Bronze layer write...")

# Create Bronze writer
bronze_writer = create_bronze_writer(spark)

# Add audit columns (tracking info)
df_with_audit = bronze_writer.add_audit_columns(
    df=df_customers,
    source_system="demo_system"
)

print("\nAudit columns added:")
print("  • _ingestion_timestamp (when data was loaded)")
print("  • _source_system (where data came from)")
print("  • _ingestion_date (date partition)\n")

# Create database if not exists
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.raw")

# Write to Bronze table
write_metrics = bronze_writer.write_to_bronze(
    df=df_with_audit,
    target_table="customers_demo",
    catalog="bronze",
    schema="raw",
    mode="overwrite",  # Overwrite for demo purposes
    enable_optimize=False
)

print("✅ Data saved to Bronze layer!")
print(f"Table location: bronze.raw.customers_demo")
print(f"Records written: {write_metrics.get('records_written', 'N/A')}")

logger.info("Bronze write completed")
print("\n➡️ Ready for data quality checks!")

In [0]:
# STEP 4: DATA QUALITY CHECKS
# This validates the data to ensure it's correct
# Checks: null values, duplicates, email format

from lakeforge.dq import create_dq_engine

print("Running data quality validations...\n")

dq_engine = create_dq_engine(spark)

# Read the data we just saved
df_bronze = spark.table("bronze.raw.customers_demo")

# Define validation rules
validation_rules = [
    {
        "rule_name": "Customer ID must not be null",
        "rule_type": "null_check",
        "column": "customer_id",
        "threshold": 0.0  # 0% nulls allowed
    },
    {
        "rule_name": "Email must not be null",
        "rule_type": "null_check",
        "column": "email",
        "threshold": 5.0  # Allow up to 5% nulls
    },
    {
        "rule_name": "Customer ID must be unique",
        "rule_type": "duplicate_check",
        "columns": ["customer_id"]
    },
    {
        "rule_name": "Email format validation",
        "rule_type": "regex_check",
        "column": "email",
        "pattern": "^[\\w\\.-]+@[\\w\\.-]+\\.\\w+$"
    }
]

# Run validations
dq_results = dq_engine.validate_dataframe(
    df=df_bronze,
    rules=validation_rules
)

# Display results
print("="*60)
print("DATA QUALITY RESULTS")
print("="*60)
print(f"Total Rules: {dq_results['rules_executed']}")
print(f"✅ Passed: {dq_results['rules_passed']}")
print(f"❌ Failed: {dq_results['rules_failed']}")
print(f"\nQuality Score: {dq_results['rules_passed']}/{dq_results['rules_executed']}")

if dq_results['rules_failed'] > 0:
    print("\n⚠️ Some checks failed. Review details below:")
    for result in dq_results['rule_results']:
        if not result['passed']:
            print(f"  ❌ {result['rule_name']}: {result['message']}")
else:
    print("\n✅ All data quality checks passed!")

print("\n➡️ Ready for trust validations!")

In [0]:
# STEP 5: TRUST ENGINE VALIDATIONS
# This ensures data integrity and consistency
# Checks: row counts, duplicates, null spikes

from lakeforge.trust_engine import create_trust_engine

print("Running trust validations...\n")

trust_engine = create_trust_engine(spark)

# For demo, we compare the data against itself
df_source = df_customers  # Original data
df_target = spark.table("bronze.raw.customers_demo")  # Saved data

# Define trust validations
trust_checks = [
    {
        "type": "row_count",
        "params": {
            "source_df": df_source,
            "target_df": df_target,
            "tolerance_percent": 5.0
        }
    },
    {
        "type": "duplicate_explosion",
        "params": {
            "source_df": df_source,
            "target_df": df_target,
            "key_columns": ["customer_id"],
            "max_explosion_ratio": 1.2
        }
    },
    {
        "type": "null_spike",
        "params": {
            "df": df_target,
            "column": "email",
            "historical_null_rate": 0.0,
            "max_spike_percent": 10.0
        }
    }
]

# Run trust validations
trust_results = trust_engine.run_trust_validations(trust_checks)

# Display results
print("="*60)
print("TRUST VALIDATION RESULTS")
print("="*60)
print(f"Total Validations: {trust_results['total_validations']}")
print(f"✅ Passed: {trust_results['passed_count']}")
print(f"❌ Failed: {trust_results['failed_count']}")
print(f"\nTrust Score: {trust_results['trust_score']:.1f}%")

if trust_results['failed_count'] > 0:
    print("\n⚠️ Some trust checks failed:")
    for validation in trust_results['validations']:
        if not validation['passed']:
            print(f"  ❌ {validation['validation_type']}: {validation['message']}")
else:
    print("\n✅ All trust validations passed!")

print("\n➡️ Ready to generate final report!")

In [0]:
# STEP 6: GENERATE TRUST REPORT
# This creates a comprehensive report combining all checks
# Output: JSON summary (displayed below)

from lakeforge.reporting import create_report_generator
import json

print("Generating trust report...\n")

report_gen = create_report_generator()

# Create schema drift placeholder (no actual drift to check in demo)
schema_drift = {"has_drift": False, "drift_score": 0.0}

# Generate comprehensive trust report
trust_report = report_gen.generate_trust_report(
    dq_results=dq_results,
    trust_results=trust_results,
    schema_drift_results=schema_drift,
    pipeline_name="Simple Customer Pipeline",
    report_title="Customer Data Trust Report"
)

# Display report summary
print("="*60)
print("FINAL TRUST REPORT")
print("="*60)
print(f"Pipeline: {trust_report['pipeline_name']}")
print(f"Generated: {trust_report['report_timestamp']}")
print(f"\n⭐ Overall Trust Score: {trust_report['overall_trust_score']:.1f}%")
print(f"Trust Level: {trust_report['trust_level']}")

print("\n" + "="*60)
print("DETAILED BREAKDOWN")
print("="*60)

# Data Quality Summary
print("\n1️⃣ Data Quality:")
print(f"   Rules Passed: {trust_report['data_quality']['rules_passed']}/{trust_report['data_quality']['rules_executed']}")
print(f"   Score: {(trust_report['data_quality']['rules_passed']/trust_report['data_quality']['rules_executed']*100):.1f}%")

# Trust Validations Summary
print("\n2️⃣ Trust Validations:")
print(f"   Checks Passed: {trust_report['trust_validations']['passed_count']}/{trust_report['trust_validations']['total_validations']}")
print(f"   Trust Score: {trust_report['trust_validations']['trust_score']:.1f}%")

# Schema Drift Summary
print("\n3️⃣ Schema Drift:")
print(f"   Drift Detected: {trust_report['schema_drift']['has_drift']}")
print(f"   Drift Score: {trust_report['schema_drift']['drift_score']}")

print("\n" + "="*60)

# Display full JSON report (optional)
print("\n📝 Full Report JSON:")
print(json.dumps(trust_report, indent=2, default=str))

print("\n✅ Report generated successfully!")
print("\n➡️ Pipeline complete! Review the summary above.")

# ✅ Pipeline Complete!

## What Just Happened:

1. ✅ **Sample Data Created**: 10 customer records generated
2. ✅ **Bronze Layer**: Data saved with audit tracking to `bronze.raw.customers_demo`
3. ✅ **Data Quality**: Validated nulls, duplicates, and email formats
4. ✅ **Trust Engine**: Checked row counts, explosions, and null spikes
5. ✅ **Trust Report**: Generated comprehensive quality report

---

## Understanding the Trust Score:

| Score | Level | Meaning |
|-------|-------|--------|
| 95-100% | 🟢 EXCELLENT | Production-ready, high confidence |
| 85-94% | 🟡 GOOD | Minor issues, generally safe |
| 70-84% | 🟠 ACCEPTABLE | Review warnings, proceed with caution |
| 50-69% | 🟠 POOR | Significant issues, needs attention |
| <50% | 🔴 CRITICAL | Do not use, fix issues first |

---

## Next Steps:

### For Your Own Data:
1. Replace Step 2 with your actual data source (CSV, Excel, API)
2. Adjust validation rules in Step 4 to match your data requirements
3. Configure trust checks in Step 5 based on your quality standards

### View Your Data:
```sql
-- Run this in a SQL cell to see your data:
SELECT * FROM bronze.raw.customers_demo LIMIT 100;
```

### Clean Up (Optional):
```python
# Run this to delete the demo table:
spark.sql("DROP TABLE IF EXISTS bronze.raw.customers_demo")
```

---

## Need Help?
- 📚 Check `/lakeforge/README.md` for full documentation
- 🔧 Review `/lakeforge/docs/testing/TESTING_GUIDE.md` for advanced testing
- 📧 Contact your data engineering team for support

---

**🎉 Congratulations! You've successfully run your first LakeForge pipeline!**